# Quick MEM — 1D Mechanical Earth Model Workflow (QC)

End-to-end GeomechPy MEM with **all intermediate results plotted** for QC.

**Units**
- Stresses / pressures: **psi**
- Mud window: **ppg** (equivalent mud weight)
- Conversion: `EMW (ppg) = P (psi) / (0.052 × TVD (ft))`

Sequence: data → overburden → lithology → pore pressure → dynamic elastic → static elastic → rock strength → horizontal stress → wellbore stability.


## 0. Setup & imports


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "Project":
    REPO_ROOT = REPO_ROOT.parent.parent
elif REPO_ROOT.name == "example":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "example"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from geomechpy.overburden_stress import OverburdenStressCalculation
from geomechpy.pore_pressure import PorePressureCalculation
from geomechpy.elastic_properties import ElasticPropertiesConverter
from geomechpy.static_elastic_properties import StaticElasticPropertiesConverter
from geomechpy.rock_strength import RockStrengthPropertiesConverter
from geomechpy.stress_calculations import HorizontalStressesCalculation
from geomechpy.wellbore_stability import WellboreStabilityCalculation
from geomechpy.toolbox import rotate_stress_to_shmax, rotate_nev_to_toh
from toolbox import determine_lithology_array

def psi_to_ppg(pressure_psi, tvd_ft):
    """Equivalent mud weight (ppg) from pressure (psi) and TVD (ft)."""
    tvd = np.asarray(tvd_ft, dtype=float)
    p = np.asarray(pressure_psi, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        emw = p / (0.052 * tvd)
    return emw

print("Imports OK — repo root:", REPO_ROOT)


## 1. Data input (synthetic logs)

Onshore well, TVD 5 000–8 000 ft. Curves mimic typical LAS inputs for a 1-D MEM.


In [ ]:
np.random.seed(42)
n = 61
tvd = np.linspace(5000.0, 8000.0, n)

gr = 40 + 40 * np.sin(np.linspace(0, 4 * np.pi, n)) + np.random.normal(0, 5, n)
gr = np.clip(gr, 15, 140)
rhob = 2.35 + 0.00004 * (tvd - 5000) + np.random.normal(0, 0.02, n)
dtco = 90 - 0.005 * (tvd - 5000) + np.random.normal(0, 2, n)
dtsh = 160 - 0.008 * (tvd - 5000) + np.random.normal(0, 3, n)
nphi = 0.18 - 0.00002 * (tvd - 5000) + np.random.normal(0, 0.01, n)
nphi = np.clip(nphi, 0.05, 0.30)

coal_flag = [False] * n
limestone_flag = [False] * n
coal_flag[20] = True
limestone_flag[45] = True

df = pd.DataFrame({
    "TVD": tvd,
    "GR": gr,
    "RHOB": rhob,
    "DTCO": dtco,
    "DTSH": dtsh,
    "NPHI": nphi,
    "COAL": coal_flag,
    "LIME": limestone_flag,
})
df.head()


### QC plot — input logs


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 8), sharey=True)
axes[0].plot(df["GR"], df["TVD"], "g-")
axes[0].axvline(75, color="k", ls="--", lw=0.8, label="GR cutoff 75")
axes[0].set_xlabel("GR (gAPI)"); axes[0].invert_yaxis(); axes[0].set_ylabel("TVD (ft)")
axes[0].set_title("Gamma Ray"); axes[0].legend(fontsize=7)
axes[1].plot(df["RHOB"], df["TVD"], "b-"); axes[1].set_xlabel("RHOB (g/cm³)"); axes[1].set_title("Density")
axes[2].plot(df["DTCO"], df["TVD"], "r-", label="DTCO"); axes[2].plot(df["DTSH"], df["TVD"], "m-", label="DTSH")
axes[2].set_xlabel("µs/ft"); axes[2].legend(fontsize=7); axes[2].set_title("Slowness")
axes[3].plot(df["NPHI"], df["TVD"], "c-"); axes[3].set_xlabel("NPHI"); axes[3].set_title("Porosity")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle("QC-1 Input logs", fontsize=12)
plt.tight_layout(); plt.show()


## 2. Overburden stress (onshore) — psi


In [ ]:
AIR_GAP = 30.0
LITHOSTATIC_GRAD = 1.05  # psi/ft

df["SV"] = OverburdenStressCalculation.calculate_overburden_stress_onshore_array(
    tvd=df["TVD"].tolist(),
    lithostatic_gradient=LITHOSTATIC_GRAD,
    air_gap=AIR_GAP,
)
df[["TVD", "SV"]].head()


## 3. Lithology (mechanical stratigraphy)


In [ ]:
df["LITH"] = determine_lithology_array(
    gamma_ray=df["GR"].tolist(),
    gr_threshold=75.0,
    coal_flag=df["COAL"].tolist(),
    limestone_flag=df["LIME"].tolist(),
)
df["LITH_NAME"] = df["LITH"].map({0: "Sand", 1: "Shale", 2: "Limestone", 6: "Coal"})
print(df["LITH_NAME"].value_counts())


### QC plot — overburden + lithology


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 8), sharey=True)
axes[0].plot(df["SV"], df["TVD"], "k-", lw=2)
axes[0].set_xlabel("Sv (psi)"); axes[0].invert_yaxis(); axes[0].set_ylabel("TVD (ft)")
axes[0].set_title("Overburden stress")
axes[1].plot(df["GR"], df["TVD"], "g-"); axes[1].axvline(75, color="k", ls="--", lw=0.8)
axes[1].set_xlabel("GR (gAPI)"); axes[1].set_title("GR vs cutoff")
colors = {0: "gold", 1: "gray", 2: "cyan", 6: "black"}
for code, name in [(0, "Sand"), (1, "Shale"), (2, "Limestone"), (6, "Coal")]:
    m = df["LITH"] == code
    if m.any():
        axes[2].scatter(df.loc[m, "LITH"], df.loc[m, "TVD"], c=colors[code], s=14, label=name)
axes[2].set_xlabel("LITH code"); axes[2].set_xticks([0, 1, 2, 6]); axes[2].legend(fontsize=7)
axes[2].set_title("Lithology")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle("QC-2 Overburden + lithology", fontsize=12)
plt.tight_layout(); plt.show()


## 4. Pore pressure (onshore) — psi


In [ ]:
PP_GRAD = 0.465  # psi/ft

df["PP"] = PorePressureCalculation.calculate_pore_pressure_onshore_array(
    tvd=df["TVD"].tolist(),
    formation_pore_pressure_gradient=PP_GRAD,
    air_gap=AIR_GAP,
)
df[["TVD", "PP", "SV"]].head()


### QC plot — pore pressure vs overburden (psi)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 8))
ax.plot(df["SV"], df["TVD"], "k-", lw=2, label="Sv")
ax.plot(df["PP"], df["TVD"], "c-", lw=2, label="Pp")
ax.invert_yaxis(); ax.set_ylabel("TVD (ft)"); ax.set_xlabel("Pressure (psi)")
ax.set_title("QC-3 Pore pressure vs overburden"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 5. Dynamic elastic properties

From DTCO / DTSH (µs/ft) + bulk density. Moduli returned in Pa, then converted to Mpsi for static correlations.


In [ ]:
density_kgm3 = (df["RHOB"] * 1000).tolist()

dyn_props = ElasticPropertiesConverter.convert_dynamic_elastic_properties_from_slowness_array(
    p_wave_slowness=df["DTCO"].tolist(),
    s_wave_slowness=df["DTSH"].tolist(),
    density=density_kgm3,
)

df["YME_DYN_Pa"] = [p.youngs_modulus for p in dyn_props]
df["PR_DYN"] = [p.poissons_ratio for p in dyn_props]
df["G_DYN_Pa"] = [p.shear_modulus for p in dyn_props]

PA_TO_MPSI = 1.0 / 6.894757e9
df["YME_DYN_Mpsi"] = df["YME_DYN_Pa"] * PA_TO_MPSI
df[["TVD", "YME_DYN_Mpsi", "PR_DYN"]].head()


## 6. Static elastic properties


In [ ]:
df["YME_STA_Mpsi"] = StaticElasticPropertiesConverter.dyn2sta_yme_bradord_array(
    yme_dyn=df["YME_DYN_Mpsi"].tolist()
)
df["PR_STA"] = StaticElasticPropertiesConverter.dyn2sta_poissons_ratio_array(
    pr_dyn=df["PR_DYN"].tolist(),
    multiplier=1.0,
)
df["BIOT"] = [
    StaticElasticPropertiesConverter.biot_coefficient_constant_law(1.0)
    for _ in range(len(df))
]
df[["TVD", "YME_STA_Mpsi", "PR_STA", "BIOT"]].head()


### QC plot — dynamic vs static elastic


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 8), sharey=True)
axes[0].plot(df["YME_DYN_Mpsi"], df["TVD"], "b-", label="Dynamic")
axes[0].plot(df["YME_STA_Mpsi"], df["TVD"], "r-", label="Static (Bradford)")
axes[0].set_xlabel("YME (Mpsi)"); axes[0].invert_yaxis(); axes[0].set_ylabel("TVD (ft)")
axes[0].set_title("Young's modulus"); axes[0].legend(fontsize=8)
axes[1].plot(df["PR_DYN"], df["TVD"], "b-", label="Dynamic")
axes[1].plot(df["PR_STA"], df["TVD"], "r-", label="Static")
axes[1].set_xlabel("Poisson's ratio"); axes[1].set_title("Poisson's ratio"); axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle("QC-4 Dynamic vs static elastic properties", fontsize=12)
plt.tight_layout(); plt.show()


## 7. Rock strength (UCS, TSTR, FANG)


In [ ]:
df["UCS"] = RockStrengthPropertiesConverter.convert_yme_sta_to_ucs_plumb_array(
    yme_sta=df["YME_STA_Mpsi"].tolist()
)
df["TSTR"] = RockStrengthPropertiesConverter.convert_ucs_to_tstr_array(
    ucs=df["UCS"].tolist(),
    multiplier=0.15,
)
df["FANG"] = RockStrengthPropertiesConverter.convert_friction_angle_lal_array(
    dtco=df["DTCO"].tolist()
)
df[["TVD", "UCS", "TSTR", "FANG"]].head()


### QC plot — rock strength


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 8), sharey=True)
axes[0].plot(df["UCS"], df["TVD"], "brown"); axes[0].set_xlabel("UCS (psi)")
axes[0].invert_yaxis(); axes[0].set_ylabel("TVD (ft)"); axes[0].set_title("UCS (Plumb)")
axes[1].plot(df["TSTR"], df["TVD"], "teal"); axes[1].set_xlabel("TSTR (psi)"); axes[1].set_title("Tensile strength")
axes[2].plot(df["FANG"], df["TVD"], "navy"); axes[2].set_xlabel("FANG (deg)"); axes[2].set_title("Friction angle (Lal)")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle("QC-5 Rock strength", fontsize=12)
plt.tight_layout(); plt.show()


## 8. Horizontal stresses (poroelastic) — psi

Plus example stress-tensor rotation into NEV at mid-depth.


In [ ]:
hs_list = HorizontalStressesCalculation.calculate_poroelastic_horizontal_stresses_array(
    overburden_stress=df["SV"].tolist(),
    pore_pressure=df["PP"].tolist(),
    poisson_ratio=df["PR_STA"].tolist(),
    youngs_modulus=df["YME_STA_Mpsi"].tolist(),
    biot_coefficient=df["BIOT"].tolist(),
    EX=0.0001,
    EY=0.0005,
)
df["SHMIN"] = [h.shmin for h in hs_list]
df["SHMAX"] = [h.shmax for h in hs_list]
df["Q_FACTOR"] = [h.q_factor for h in hs_list]
df["SHMAX_SHMIN_RATIO"] = [h.shmax_shmin_ratio for h in hs_list]

mid = len(df) // 2
SHMAX_AZIMUTH = 45.0
stress_nev = rotate_stress_to_shmax(
    shmin=df["SHMIN"].iloc[mid],
    shmax=df["SHMAX"].iloc[mid],
    svert=df["SV"].iloc[mid],
    shmax_azimuth=SHMAX_AZIMUTH,
)
print("Stress tensor NEV at mid-depth (psi):")
print(np.round(stress_nev, 1))
df[["TVD", "SHMIN", "SHMAX", "Q_FACTOR"]].head()


### QC plot — stress profile (psi)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 8), sharey=True)
axes[0].plot(df["SV"], df["TVD"], "k-", lw=2, label="Sv")
axes[0].plot(df["SHMAX"], df["TVD"], "r-", label="SHmax")
axes[0].plot(df["SHMIN"], df["TVD"], "b-", label="Shmin")
axes[0].plot(df["PP"], df["TVD"], "c-", label="Pp")
axes[0].set_xlabel("Stress (psi)"); axes[0].invert_yaxis(); axes[0].set_ylabel("TVD (ft)")
axes[0].set_title("Stress profile"); axes[0].legend(fontsize=8)
axes[1].plot(df["Q_FACTOR"], df["TVD"], "purple")
axes[1].axvline(1, color="k", ls="--", lw=0.7); axes[1].axvline(2, color="k", ls="--", lw=0.7)
axes[1].set_xlabel("q-factor"); axes[1].set_title("Stress regime")
axes[2].plot(df["SHMAX_SHMIN_RATIO"], df["TVD"], "orange")
axes[2].set_xlabel("SHmax / Shmin"); axes[2].set_title("Anisotropy")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle("QC-6 Horizontal stresses (psi)", fontsize=12)
plt.tight_layout(); plt.show()


## 9. Wellbore stability — pressures in psi, mud window in ppg

Breakout (Mohr–Coulomb) and breakdown (tensile). EMW converted with `P / (0.052 \u00d7 TVD)`.


In [ ]:
df["PW_BREAKOUT"] = (
    WellboreStabilityCalculation
    .calculate_breakout_calculation_vertical_well_mohr_coulomb_analytical_array(
        shmax=df["SHMAX"].tolist(),
        shmin=df["SHMIN"].tolist(),
        pprs=df["PP"].tolist(),
        overburden_stress=df["SV"].tolist(),
        ucs=df["UCS"].tolist(),
        fang=df["FANG"].tolist(),
        pr_sta=df["PR_STA"].tolist(),
    )
)
df["PW_BREAKDOWN"] = (
    WellboreStabilityCalculation
    .calculate_breakdown_calculation_vertical_well_analytical_array(
        shmax=df["SHMAX"].tolist(),
        shmin=df["SHMIN"].tolist(),
        pprs=df["PP"].tolist(),
        tstr=df["TSTR"].tolist(),
    )
)

df["EMW_BREAKOUT"] = psi_to_ppg(df["PW_BREAKOUT"], df["TVD"])
df["EMW_BREAKDOWN"] = psi_to_ppg(df["PW_BREAKDOWN"], df["TVD"])
df["EMW_PP"] = psi_to_ppg(df["PP"], df["TVD"])
df["EMW_SV"] = psi_to_ppg(df["SV"], df["TVD"])

df[["TVD", "PW_BREAKOUT", "PW_BREAKDOWN", "EMW_BREAKOUT", "EMW_BREAKDOWN"]].head()


### QC plot — mud window in ppg


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 8), sharey=True)

# Left: pressures in psi
axes[0].plot(df["PW_BREAKOUT"], df["TVD"], "r-", lw=2, label="Breakout")
axes[0].plot(df["PW_BREAKDOWN"], df["TVD"], "b-", lw=2, label="Breakdown")
axes[0].plot(df["PP"], df["TVD"], "c--", label="Pp")
axes[0].plot(df["SV"], df["TVD"], "k:", label="Sv")
axes[0].invert_yaxis(); axes[0].set_ylabel("TVD (ft)")
axes[0].set_xlabel("Pressure (psi)"); axes[0].set_title("Limits (psi)"); axes[0].legend(fontsize=8)

# Right: EMW in ppg
axes[1].plot(df["EMW_BREAKOUT"], df["TVD"], "r-", lw=2, label="Breakout")
axes[1].plot(df["EMW_BREAKDOWN"], df["TVD"], "b-", lw=2, label="Breakdown")
axes[1].plot(df["EMW_PP"], df["TVD"], "c--", label="Pp EMW")
axes[1].plot(df["EMW_SV"], df["TVD"], "k:", label="Sv EMW")
axes[1].fill_betweenx(df["TVD"], df["EMW_BREAKOUT"], df["EMW_BREAKDOWN"], alpha=0.15, color="green", label="Safe window")
axes[1].set_xlabel("EMW (ppg)"); axes[1].set_title("Mud window (ppg)"); axes[1].legend(fontsize=8)

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle("QC-7 Wellbore stability — psi & ppg", fontsize=12)
plt.tight_layout(); plt.show()


## 10. Full MEM summary plot (QC)


In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(16, 8), sharey=True)

axes[0].plot(df["GR"], df["TVD"], "g-")
axes[0].set_xlabel("GR (gAPI)"); axes[0].invert_yaxis(); axes[0].set_ylabel("TVD (ft)")

axes[1].plot(df["SV"], df["TVD"], "k-", label="Sv")
axes[1].plot(df["SHMAX"], df["TVD"], "r-", label="SHmax")
axes[1].plot(df["SHMIN"], df["TVD"], "b-", label="Shmin")
axes[1].plot(df["PP"], df["TVD"], "c-", label="Pp")
axes[1].set_xlabel("Stress (psi)"); axes[1].legend(fontsize=6)

axes[2].plot(df["YME_STA_Mpsi"], df["TVD"], "m-")
axes[2].set_xlabel("YME_sta (Mpsi)")

axes[3].plot(df["PR_STA"], df["TVD"], "orange")
axes[3].set_xlabel("PR_sta")

axes[4].plot(df["UCS"], df["TVD"], "brown")
axes[4].set_xlabel("UCS (psi)")

axes[5].plot(df["FANG"], df["TVD"], "navy")
axes[5].set_xlabel("FANG (deg)")

axes[6].plot(df["EMW_BREAKOUT"], df["TVD"], "r-", label="Breakout")
axes[6].plot(df["EMW_BREAKDOWN"], df["TVD"], "b-", label="Breakdown")
axes[6].plot(df["EMW_PP"], df["TVD"], "c--", label="Pp")
axes[6].set_xlabel("EMW (ppg)"); axes[6].legend(fontsize=6)

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle("QC-8 Full MEM summary — stresses in psi, mud window in ppg", fontsize=12)
plt.tight_layout(); plt.show()


## Done

All intermediate steps have dedicated QC plots. Stresses are in **psi**; mud-window limits are in **ppg**.

```python
df.to_csv("quick_mem_results.csv", index=False)
```
